# 02 — Baseline Models: TF-IDF + Classical ML

Building the baseline. We'll use TF-IDF features with SVM and Logistic Regression, then compare against dense embeddings.

The goal here is to have a solid number to beat. TF-IDF + LinearSVC is actually surprisingly strong — don't underestimate it.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

from src.utils.data_loader import load_20newsgroups
from src.models.tfidf_classifier import TfidfClassifier
from src.models.embedding_classifier import EmbeddingClassifier

print('ready')

In [ ]:
train_texts, train_labels, test_texts, test_labels = load_20newsgroups()
print(f'Train: {len(train_texts)} | Test: {len(test_texts)}')

In [ ]:
# --- Experiment 1: TF-IDF + SVM ---
print('Training TF-IDF + SVM...')
svm_clf = TfidfClassifier(classifier_type='svm')
svm_clf.train(train_texts, train_labels)
svm_results = svm_clf.evaluate(test_texts, test_labels)
print(f"SVM Accuracy: {svm_results['accuracy']:.4f}")

In [ ]:
# --- Experiment 2: TF-IDF + Logistic Regression ---
print('Training TF-IDF + LogReg...')
lr_clf = TfidfClassifier(classifier_type='logreg')
lr_clf.train(train_texts, train_labels)
lr_results = lr_clf.evaluate(test_texts, test_labels)
print(f"LogReg Accuracy: {lr_results['accuracy']:.4f}")

In [ ]:
# --- Experiment 3: Dense Embeddings + KNN ---
# this takes a few minutes — embedding 10k+ docs
print('Training Embedding + KNN...')
emb_clf = EmbeddingClassifier(embedding_model='all-MiniLM-L6-v2', classifier_type='knn')
emb_clf.train(train_texts, train_labels)
emb_results = emb_clf.evaluate(test_texts, test_labels)
print(f"Embedding KNN Accuracy: {emb_results['accuracy']:.4f}")

In [ ]:
# comparison table
import pandas as pd

comparison = pd.DataFrame([
    {'Model': 'TF-IDF + SVM',       'Accuracy': svm_results['accuracy']},
    {'Model': 'TF-IDF + LogReg',    'Accuracy': lr_results['accuracy']},
    {'Model': 'Embeddings + KNN',   'Accuracy': emb_results['accuracy']},
])

comparison['Accuracy'] = comparison['Accuracy'].map('{:.4f}'.format)
print(comparison.to_string(index=False))

In [ ]:
# confusion matrix for best TF-IDF model
pred_labels = svm_clf.predict(test_texts)
classes = sorted(set(test_labels))

cm = confusion_matrix(test_labels, pred_labels, labels=classes)
plt.figure(figsize=(12, 10))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
disp.plot(ax=plt.gca(), colorbar=True, cmap='Blues')
plt.title('Confusion Matrix — TF-IDF + SVM')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# save the best baseline
svm_clf.save('../models/tfidf_classifier')
emb_clf.save('../models/embedding_classifier')
print('Models saved.')